# Etapa 2 – Preparação e integração dos dados (versão Colab)
**Ciência de Dados – Projeto Final (UTFPR)** · Igor Diogo Bueno Pimenta, Guilherme Belendir Magan

Vazão no alto Iguaçu: **Porto Amazonas → Rio Negro → Fluviópolis → União da Vitória** (alvo).

**Como usar**
1. Certifique-se de colocar os arquivos originais do ONS (Grandezas Fluviométricas) em CSV diretamente na pasta `dados_ons/` (https://dados.ons.org.br/dataset/grandezas_fluviometricas).
2. Execute `Ambiente de execução → Executar tudo`.
3. O clima (Open-Meteo) é baixado automaticamente. Os feriados vêm da biblioteca `holidays`.
4. No fim, as bases em Parquet são compactadas num `.zip` e baixadas.



In [41]:
!pip -q install holidays duckdb

In [42]:
import io, json, time, hashlib, unicodedata, zipfile
from datetime import date, timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import holidays
import matplotlib.pyplot as plt

try:
    from google.colab import files as colab_files
    EM_COLAB = True
except ImportError:
    colab_files, EM_COLAB = None, False

BASE = Path("/content") if EM_COLAB else Path.cwd()
RAW_ONS = BASE / "dados_ons"; RAW_CLIMA = BASE / "openmeteo"; SAIDA = BASE / "saida"
for p in (RAW_ONS, RAW_CLIMA, SAIDA):
    p.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_columns", 60)
print("Rodando no Colab:", EM_COLAB, "| pasta:", BASE)

Rodando no Colab: True | pasta: /content


## 1. Configuração (todas as decisões num só lugar)
- **Postos:** a busca é feita primeiro pelo código e, se não achar, pelo nome (sem acento) **dentro da bacia do Iguaçu**.
  Isso evita casar com outro "Rio Negro", como o da bacia Amazônica.
- **Coordenadas:** aproximadas (sede do município ou distrito), usadas para buscar o clima. (Informação externa ao dataset)

In [43]:
POSTOS = {
    "porto_amazonas":   {"codigos": ["65035001", "65035000"], "nomes": ["PORTO AMAZONAS"],
                         "lat": -25.5447, "lon": -49.8897},
    "rio_negro":        {"codigos": ["65100001", "65100000"], "nomes": ["RIO NEGRO"],
                         "lat": -26.1003, "lon": -49.7975},
    "fluviopolis":      {"codigos": ["65220001", "65220000"], "nomes": ["FLUVIOPOLIS"],
                         "lat": -26.0300, "lon": -50.5900},
    "uniao_da_vitoria": {"codigos": ["65310000", "65310001"], "nomes": ["UNIAO DA VITORIA", "UNIAO VITORIA"],
                         "lat": -26.2303, "lon": -51.0867},
}
NOMES_BONITOS = {"porto_amazonas": "Porto Amazonas", "rio_negro": "Rio Negro",
                 "fluviopolis": "Fluviópolis", "uniao_da_vitoria": "União da Vitória"}
FILTRO_BACIA = "IGUA"
UF = "PR"
FUSO = "America/Sao_Paulo"

MAX_GAP_INTERPOLACAO = 3     # dias; lacunas maiores ficam NaN
FATOR_PICO_ISOLADO = 5.0     # valor > 5x os DOIS vizinhos -> erro de registro
DIAS_VALOR_CONSTANTE = 10    # >= 10 dias idênticos -> só marca
MIN_PCT_DIAS_MES = 0.80      # mês com < 80% dos dias válidos não recebe média
CHUVA_DIA_CHUVOSO_MM = 1.0
REF_INICIO, REF_FIM = 2000, 2015   # período para o limiar Q90 (fora dos anos de teste)

## 2. Leitura dos arquivos originais do ONS
deixe um ou mais arquivos `GRANDEZAS_FLUVIOMETRICAS_<ano>` (`.csv`, `.parquet` ou `.xlsx`) na pasta `dados_ons/`

In [44]:
arquivos_ons = sorted(p for p in RAW_ONS.iterdir() if p.suffix.lower() in (".csv", ".parquet", ".xlsx"))
assert arquivos_ons, f"Nenhum arquivo do ONS encontrado na pasta '{RAW_ONS}'. Certifique-se de colocar os arquivos lá antes de rodar."

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

log_arquivos = pd.DataFrame([{"arquivo": p.name, "MB": round(p.stat().st_size / 1e6, 2),
                              "sha256": sha256(p)} for p in arquivos_ons])
log_arquivos

,arquivo,MB,sha256
0,GRANDEZAS_FLUVIOMETRICAS_2025 (1).csv,7.24,68be92fa22a1ce2fe3ef2387354a8fa280bfa0db5b98d2...
1,GRANDEZAS_FLUVIOMETRICAS_2025 (2).csv,7.24,68be92fa22a1ce2fe3ef2387354a8fa280bfa0db5b98d2...
2,GRANDEZAS_FLUVIOMETRICAS_2026 (1).csv,6.74,1829fefbe46cb871f1cb56867d34e47161f60d058f6dca...
3,GRANDEZAS_FLUVIOMETRICAS_2026 (2).csv,6.74,1829fefbe46cb871f1cb56867d34e47161f60d058f6dca...


## 3. Leitura e formatos
- Tudo é lido como **texto** e convertido de forma explícita.
- **Números:** aceitamos `1234.5`, `1234,5` e `1.234,5`.
- **Datas:** utilizamos o formato YYYY-MM-DD, tal como esta no dicionario de dados

In [45]:
def normalizar_texto(s):
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode()
    return " ".join(s.upper().split())

def ler_arquivo_bruto(path):
    suf = path.suffix.lower()
    if suf == ".parquet":
        df = pd.read_parquet(path).astype("string")
    elif suf == ".xlsx":
        df = pd.read_excel(path, dtype=str)
    else:
        for sep in (";", ","):
            df = pd.read_csv(path, sep=sep, dtype=str, encoding="utf-8-sig", keep_default_na=False,
                             na_values=["", "NA", "NaN", "null", "NULL", "-"])
            if df.shape[1] > 2:
                break
    df.columns = [str(c).strip().lower() for c in df.columns]
    df["arquivo_origem"] = path.name
    return df

def detectar_colunas(cols):
    cols = list(cols)
    def achar(cond, obrig=True, nome=""):
        achadas = [c for c in cols if cond(c)]
        if not achadas and obrig:
            raise KeyError(f"Coluna '{nome}' não encontrada. Colunas do arquivo: {cols}")
        return achadas[0] if achadas else None
    return {
        "data": achar(lambda c: c.startswith(("din_", "dat_")) or c in ("data", "dt_medicao"), nome="data"),
        "vazao": achar(lambda c: c == "val_vazaomedia", obrig=False)
                 or achar(lambda c: "vazao" in c and c.startswith("val_"), nome="vazão"),
        "cod_posto": achar(lambda c: c.startswith("cod_") and "posto" in c, obrig=False),
        "nom_posto": achar(lambda c: c.startswith("nom_") and "posto" in c, obrig=False),
        "bacia": achar(lambda c: "bacia" in c and c.startswith("nom_"), obrig=False),
        "rio": achar(lambda c: c.startswith("nom_rio"), obrig=False),
    }

def converter_numero(s):
    s = s.astype("string").str.strip()
    tem_virgula = s.str.contains(",", regex=False, na=False)
    s = s.where(~tem_virgula, s.str.replace(".", "", regex=False).str.replace(",", ".", regex=False))
    return pd.to_numeric(s, errors="coerce")

def converter_data(s):
    s = s.astype("string").str.strip()
    # Reconhece apenas o formato YYYY-MM-DD
    fmt = "%Y-%m-%d"
    d = pd.to_datetime(s, format=fmt, errors="coerce")
    return d.dt.normalize(), fmt

bruto_todos = pd.concat([ler_arquivo_bruto(p) for p in arquivos_ons], ignore_index=True)
COLS = detectar_colunas(bruto_todos.columns)
print("Colunas do arquivo:", list(bruto_todos.columns))
print("Colunas detectadas:", json.dumps(COLS, indent=1))
bruto_todos.head()

Colunas do arquivo: ['id_postofluv', 'nom_postofluviometrico', 'val_latitude', 'val_longitude', 'nom_rio', 'nom_bacia', 'din_medicao', 'val_vazaomedia', 'val_vazaomediaincr', 'arquivo_origem']
Colunas detectadas: {
 "data": "din_medicao",
 "vazao": "val_vazaomedia",
 "cod_posto": null,
 "nom_posto": "nom_postofluviometrico",
 "bacia": "nom_bacia",
 "rio": "nom_rio"
}


,id_postofluv,nom_postofluviometrico,val_latitude,val_longitude,nom_rio,nom_bacia,din_medicao,val_vazaomedia,val_vazaomediaincr,arquivo_origem
0,61855000,FAZENDA BELA VISTA,-20.9872,-48.1239,RIO PARDO ...,BACIA DO RIO GRANDE ...,2025-01-12,224.38999938964844,0.0,GRANDEZAS_FLUVIOMETRICAS_2025 (1).csv
1,61855000,FAZENDA BELA VISTA,-20.9872,-48.1239,RIO PARDO ...,BACIA DO RIO GRANDE ...,2025-01-13,212.57000732421875,0.0,GRANDEZAS_FLUVIOMETRICAS_2025 (1).csv
2,61945080,RIO VERDE,-19.7908,-49.6533,RIO VERDE OU FEIO ...,BACIA DO RIO GRANDE ...,2025-01-23,27.93000030517578,0.0,GRANDEZAS_FLUVIOMETRICAS_2025 (1).csv
3,61855000,FAZENDA BELA VISTA,-20.9872,-48.1239,RIO PARDO ...,BACIA DO RIO GRANDE ...,2025-01-25,227.32000732421875,0.0,GRANDEZAS_FLUVIOMETRICAS_2025 (1).csv
4,61945080,RIO VERDE,-19.7908,-49.6533,RIO VERDE OU FEIO ...,BACIA DO RIO GRANDE ...,2025-02-05,57.22999954223633,0.0,GRANDEZAS_FLUVIOMETRICAS_2025 (1).csv


### Seleção dos 4 postos

In [46]:
df = bruto_todos
df["_nome"] = df[COLS["nom_posto"]].map(normalizar_texto) if COLS["nom_posto"] else ""
# Remove o sufixo -SIMEPAR dos nomes
df["_nome"] = df["_nome"].str.replace("-SIMEPAR", "", regex=False).str.strip()
df["_cod"] = df[COLS["cod_posto"]].astype("string").str.strip() if COLS["cod_posto"] else ""
col_bacia = COLS["bacia"] or COLS["rio"]
df["_bacia"] = df[col_bacia].map(normalizar_texto) if col_bacia else ""

partes, mapeamento = [], {}
for posto, meta in POSTOS.items():
    sel, via = df["_cod"].isin(meta["codigos"]), "código"
    if not sel.any():
        sel, via = df["_nome"].isin(meta["nomes"]), "nome"
        if col_bacia and sel.any():
            sel &= df["_bacia"].str.contains(FILTRO_BACIA)
    if not sel.any():
        candidatos = sorted(df.loc[df["_bacia"].str.contains(FILTRO_BACIA), "_nome"].unique())
        raise ValueError(f"Posto '{posto}' não encontrado. Postos da bacia do Iguaçu no arquivo: {candidatos[:40]}"
                         " -> ajuste POSTOS na seção 1.")
    sub = df.loc[sel].copy()
    if COLS["cod_posto"] and sub["_cod"].nunique() > 1:
        raise ValueError(f"'{posto}' casou com mais de um código: {sorted(sub['_cod'].unique())}. Ajuste POSTOS.")
    sub["posto"] = posto
    partes.append(sub)
    mapeamento[posto] = {"via": via, "codigos": sorted(sub["_cod"].dropna().unique()),
                         "nomes": sorted(sub["_nome"].dropna().unique()), "linhas": int(sel.sum())}

ons = pd.concat(partes, ignore_index=True)
ons["data"] = pd.NaT
formatos = {}
for arq, idx in ons.groupby("arquivo_origem").groups.items():
    ons.loc[idx, "data"], formatos[arq] = converter_data(ons.loc[idx, COLS["data"]])
ons["data"] = pd.to_datetime(ons["data"])
ons["vazao_bruta"] = converter_numero(ons[COLS["vazao"]])
print("Formatos de data encontrados:", set(formatos.values()))
print("Datas inválidas:", int(ons["data"].isna().sum()),
      "| vazões não numéricas:", int((ons["vazao_bruta"].isna() & ons[COLS["vazao"]].notna()).sum()))
ons = (ons.rename(columns={"_cod": "cod_posto", "_nome": "nom_posto"})
          [["data", "posto", "cod_posto", "nom_posto", "vazao_bruta", "arquivo_origem"]]
          .dropna(subset=["data"]).sort_values(["posto", "data"]).reset_index(drop=True))
INICIO = ons["data"].min().strftime("%Y-01-01")
FIM = ons["data"].max().strftime("%Y-%m-%d")
print("Período:", INICIO, "a", FIM)
pd.DataFrame(mapeamento).T

Formatos de data encontrados: {'%Y-%m-%d'}
Datas inválidas: 0 | vazões não numéricas: 0
Período: 2025-01-01 a 2026-11-16


,via,codigos,nomes,linhas
porto_amazonas,nome,[],[PORTO AMAZONAS],34
rio_negro,nome,[],[RIO NEGRO],38
fluviopolis,nome,[],[FLUVIOPOLIS],40
uniao_da_vitoria,nome,[],[UNIAO DA VITORIA],1370


## 4. Tratamento da vazão

Nenhum valor é apagado sem registro: a coluna `vazao_bruta` guarda o valor original, `vazao` guarda o valor tratado e `flag` indica o que aconteceu em cada dia.

Linhas duplicadas são removidas; Se o mesmo posto tiver dois valores diferentes no mesmo dia, usamos a mediana e marcamos o dia como `duplicata_conflitante`; Os dias que não vieram no arquivo são incluídos com valor (`ausente`);

Vazões negativas ou iguais a zero são consideradas inválidas; Também descartamos picos ou quedas que duram um único dia e são mais de 5 vezes maiores ou menores que os dois dias vizinhos; Sequências de 10 dias ou mais com o mesmo valor são apenas marcadas (`suspeito_constante`).

Por fim, lacunas de até 3 dias são preenchidas por interpolação linear, já que a vazão varia de forma suave. Lacunas maiores ficam vazias, para não inventar cheias ou estiagens. As cheias reais nunca são removidas, porque são justamente o objeto da P1.

In [47]:
rel = {}
# 1) duplicatas
n0 = len(ons)
v = ons.drop_duplicates(["posto", "data", "vazao_bruta"])
rel["duplicatas_exatas"] = n0 - len(v)
v = (v.groupby(["posto", "data"])
      .agg(vazao_bruta=("vazao_bruta", "median"), n_valores=("vazao_bruta", "nunique"),
           cod_posto=("cod_posto", "first"), nom_posto=("nom_posto", "first"),
           arquivo_origem=("arquivo_origem", "last"))
      .reset_index())
v["conflito"] = v["n_valores"] > 1
rel["dias_com_valores_conflitantes"] = int(v["conflito"].sum())

# 2) calendário completo
idx = pd.MultiIndex.from_product([list(POSTOS), pd.date_range(INICIO, FIM, freq="D")], names=["posto", "data"])
v = v.drop(columns="n_valores").set_index(["posto", "data"]).reindex(idx).reset_index()
v["veio_no_arquivo"] = v["arquivo_origem"].notna()
v["conflito"] = v["conflito"].fillna(False).astype(bool)
for c in ("cod_posto", "nom_posto"):
    v[c] = v.groupby("posto")[c].transform(lambda s: s.ffill().bfill())

# 3) valores suspeitos
v["vazao"] = v["vazao_bruta"].astype(float)
v["flag"] = np.where(v["veio_no_arquivo"] & v["vazao_bruta"].notna(), "ok", "ausente")
v.loc[v["conflito"], "flag"] = "duplicata_conflitante"
neg, zero = v["vazao"] < 0, v["vazao"] == 0
v.loc[neg, ["vazao", "flag"]] = [np.nan, "negativo"]
v.loc[zero, ["vazao", "flag"]] = [np.nan, "zero"]
ant = v.groupby("posto")["vazao"].shift(1)
prox = v.groupby("posto")["vazao"].shift(-1)
pico = (v["vazao"] > FATOR_PICO_ISOLADO * ant) & (v["vazao"] > FATOR_PICO_ISOLADO * prox)
vale = (v["vazao"] * FATOR_PICO_ISOLADO < ant) & (v["vazao"] * FATOR_PICO_ISOLADO < prox)
v.loc[pico, ["vazao", "flag"]] = [np.nan, "pico_isolado"]
v.loc[vale, ["vazao", "flag"]] = [np.nan, "queda_isolada"]
def seq_constante(s, n=DIAS_VALOR_CONSTANTE):
    g = (s != s.shift()).cumsum()
    return (s.groupby(g).transform("size") >= n) & s.notna()
v["suspeito_constante"] = v.groupby("posto")["vazao"].transform(seq_constante).astype(bool)
rel.update(negativos=int(neg.sum()), zeros=int(zero.sum()), picos_isolados=int(pico.sum()),
           quedas_isoladas=int(vale.sum()), dias_em_sequencia_constante=int(v["suspeito_constante"].sum()))

# 4) interpolação só de lacunas curtas
antes = v["vazao"].isna()
v["vazao"] = v.groupby("posto")["vazao"].transform(
    lambda s: s.interpolate(limit=MAX_GAP_INTERPOLACAO, limit_area="inside"))
grupo = (antes != antes.groupby(v["posto"]).shift()).cumsum()
tam_lacuna = antes.groupby(grupo).transform("size").where(antes, 0)
v.loc[antes & (tam_lacuna > MAX_GAP_INTERPOLACAO), "vazao"] = np.nan
v["interpolado"] = antes & v["vazao"].notna()
rel.update(dias_interpolados=int(v["interpolado"].sum()), dias_ainda_ausentes=int(v["vazao"].isna().sum()))

vazao = v[["data", "posto", "cod_posto", "nom_posto", "vazao_bruta", "vazao", "flag",
           "interpolado", "suspeito_constante", "veio_no_arquivo", "arquivo_origem"]]
print(json.dumps(rel, indent=1, ensure_ascii=False))

qualidade = pd.crosstab(vazao["posto"], vazao["flag"])
qualidade["interpolados"] = vazao.groupby("posto")["interpolado"].sum()
qualidade["ausentes_finais"] = vazao.groupby("posto")["vazao"].apply(lambda s: s.isna().sum())
qualidade["cobertura_final_%"] = (100 * (1 - qualidade["ausentes_finais"] / vazao.groupby("posto").size())).round(2)
qualidade.loc[list(POSTOS)]

{
 "duplicatas_exatas": 741,
 "dias_com_valores_conflitantes": 0,
 "negativos": 0,
 "zeros": 0,
 "picos_isolados": 0,
 "quedas_isoladas": 0,
 "dias_em_sequencia_constante": 0,
 "dias_interpolados": 2,
 "dias_ainda_ausentes": 1997
}


/tmp/ipykernel_525/3778429657.py:18: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  v["conflito"] = v["conflito"].fillna(False).astype(bool)


flag,ausente,ok,interpolados,ausentes_finais,cobertura_final_%
posto,,,,,
porto_amazonas,668,17,2,666,2.77
rio_negro,666,19,0,666,2.77
fluviopolis,665,20,0,665,2.92
uniao_da_vitoria,0,685,0,0,100.00


Porcentagem de dias sem vazão (depois do tratamento) por ano e posto:

In [48]:
(vazao.assign(ano=vazao["data"].dt.year, falta=vazao["vazao"].isna())
      .pivot_table(index="ano", columns="posto", values="falta", aggfunc="mean")
      .mul(100).round(1)[list(POSTOS)])

posto,porto_amazonas,rio_negro,fluviopolis,uniao_da_vitoria
ano,,,,
2025,100.0,100.0,100.0,0.0
2026,94.1,94.1,93.8,0.0


## 5. Clima – Open-Meteo Historical Weather API
- **Escala geográfica:** um ponto por posto. O Open-Meteo usa a célula da grade de reanálise mais próxima
  (as coordenadas usadas ficam em `lat_grade`/`lon_grade`), então o valor representa a área ao redor do posto.
- **Escala temporal:** dados diários no fuso `America/Sao_Paulo`, alinhados com o dia do ONS.
- **Limpeza:** valores fora de faixas físicas (chuva < 0 ou > 400 mm/dia, temperatura fora de −15…45 °C) e `tmin > tmax` viram NaN.

In [49]:
fim_clima = min(pd.Timestamp(FIM).date(), date.today() - timedelta(days=6)).isoformat()
partes = []
for posto, meta in POSTOS.items():
    cache = RAW_CLIMA / f"openmeteo_{posto}_{INICIO}_{fim_clima}.json"
    if not cache.exists():
        params = {"latitude": meta["lat"], "longitude": meta["lon"], "start_date": INICIO, "end_date": fim_clima,
                  "daily": "precipitation_sum,temperature_2m_mean,temperature_2m_max,temperature_2m_min",
                  "timezone": FUSO}
        for tentativa in range(3):
            try:
                r = requests.get("https://archive-api.open-meteo.com/v1/archive", params=params, timeout=120)
                r.raise_for_status(); cache.write_text(r.text); break
            except requests.RequestException as e:
                print(posto, "falhou:", e); time.sleep(5 * (tentativa + 1))
    j = json.loads(cache.read_text())
    d = pd.DataFrame(j["daily"]).rename(columns={
        "time": "data", "precipitation_sum": "chuva_mm", "temperature_2m_mean": "tmed_c",
        "temperature_2m_max": "tmax_c", "temperature_2m_min": "tmin_c"})
    d["data"] = pd.to_datetime(d["data"])
    d["posto"], d["lat_grade"], d["lon_grade"] = posto, j.get("latitude"), j.get("longitude")
    partes.append(d)
clima = pd.concat(partes, ignore_index=True).drop_duplicates(["data", "posto"], keep="last")

rel_clima = {}
for col, (lo, hi) in {"chuva_mm": (0, 400), "tmed_c": (-10, 40), "tmax_c": (-10, 45), "tmin_c": (-15, 35)}.items():
    fora = ~clima[col].between(lo, hi) & clima[col].notna()
    rel_clima[f"{col}_fora_da_faixa"] = int(fora.sum()); clima.loc[fora, col] = np.nan
inv = clima["tmin_c"] > clima["tmax_c"]
rel_clima["tmin_maior_que_tmax"] = int(inv.sum()); clima.loc[inv, ["tmin_c", "tmax_c"]] = np.nan
clima = clima[clima["data"].between(INICIO, FIM)]
print(rel_clima)
clima.groupby("posto")[["lat_grade", "lon_grade"]].first()

{'chuva_mm_fora_da_faixa': 0, 'tmed_c_fora_da_faixa': 0, 'tmax_c_fora_da_faixa': 0, 'tmin_c_fora_da_faixa': 0, 'tmin_maior_que_tmax': 0}


,lat_grade,lon_grade
posto,,
fluviopolis,-26.045694,-50.612670
porto_amazonas,-25.553602,-49.934845
rio_negro,-26.115993,-49.780945
uniao_da_vitoria,-26.256590,-51.075745


## 6. Feriados e calendário
Os 4 postos ficam no **Paraná**, então os feriados são associados **só pela data**: nacionais, estaduais do PR e pontos
facultativos (Carnaval, Corpus Christi…). Feriados municipais não estão na biblioteca e ficaram de fora. Como a hipótese da
Etapa 1 é que a vazão natural não depende de feriados, eles entram como controle; o efeito será medido nas etapas seguintes.

In [50]:
anos = list(range(int(INICIO[:4]), int(FIM[:4]) + 1))
nac = holidays.country_holidays("BR", years=anos)
est = holidays.country_holidays("BR", subdiv=UF, years=anos)
try:
    opc = holidays.country_holidays("BR", subdiv=UF, years=anos, categories=("optional",))
except Exception:
    opc = {}

cal = pd.DataFrame({"data": pd.date_range(INICIO, FIM, freq="D")})
dd = cal["data"].dt.date
cal["ano"], cal["mes"] = cal["data"].dt.year, cal["data"].dt.month
cal["dia_semana"], cal["dia_ano"] = cal["data"].dt.dayofweek, cal["data"].dt.dayofyear
cal["fim_de_semana"] = cal["dia_semana"] >= 5
cal["estacao"] = cal["mes"].map({12: "verao", 1: "verao", 2: "verao", 3: "outono", 4: "outono", 5: "outono",
                                 6: "inverno", 7: "inverno", 8: "inverno",
                                 9: "primavera", 10: "primavera", 11: "primavera"})
cal["feriado_nacional"] = dd.map(lambda x: x in nac)
cal["feriado_estadual"] = dd.map(lambda x: x in est and x not in nac)
cal["ponto_facultativo"] = dd.map(lambda x: x in opc and x not in est)
cal["nome_feriado"] = dd.map(lambda x: est.get(x) or (opc.get(x) if x in opc else None) or "")
cal["feriado"] = cal["feriado_nacional"] | cal["feriado_estadual"]
cal["dia_util"] = ~(cal["fim_de_semana"] | cal["feriado"])
folga = cal["feriado"] | cal["fim_de_semana"]
cal["ponte"] = cal["dia_util"] & folga.shift(1, fill_value=False) & folga.shift(-1, fill_value=False)
cal.loc[cal["feriado"] | cal["ponto_facultativo"], ["data", "nome_feriado", "feriado_nacional",
                                                    "feriado_estadual", "ponto_facultativo"]].tail(12)

,data,nome_feriado,feriado_nacional,feriado_estadual,ponto_facultativo
411,2026-02-16,Carnival,False,False,True
412,2026-02-17,Carnival,False,False,True
413,2026-02-18,Ash Wednesday,False,False,True
457,2026-04-03,Good Friday,True,False,False
475,2026-04-21,Tiradentes' Day,True,False,False
485,2026-05-01,Worker's Day,True,False,False
519,2026-06-04,Corpus Christi,False,False,True
614,2026-09-07,Independence Day,True,False,False
649,2026-10-12,Our Lady of Aparecida,True,False,False
665,2026-10-28,Public Servant's Day,False,False,True


## 7. Integração – base diária
Uma linha por dia, com o calendário, a vazão tratada de cada posto (`vazao_<posto>` + `flag_<posto>`) e o clima de cada ponto.
- `chuva_bacia` = **média simples** dos 4 pontos (como previsto na Etapa 1); `tmed/tmax/tmin_bacia` = média dos 4 pontos.
- Se faltar o clima de 2 ou mais pontos no dia, a média da bacia fica NaN.


In [51]:
w_v = vazao.pivot(index="data", columns="posto", values="vazao").add_prefix("vazao_")
w_f = vazao.pivot(index="data", columns="posto", values="flag").add_prefix("flag_")
w_c = clima.pivot(index="data", columns="posto", values=["chuva_mm", "tmed_c", "tmax_c", "tmin_c"])
curto = {"chuva_mm": "chuva", "tmed_c": "tmed", "tmax_c": "tmax", "tmin_c": "tmin"}
w_c.columns = [f"{curto[var]}_{p}" for var, p in w_c.columns]
for var in curto.values():
    cols = [f"{var}_{p}" for p in POSTOS]
    w_c[f"{var}_bacia"] = w_c[cols].mean(axis=1)
    w_c.loc[w_c[cols].notna().sum(axis=1) < 3, f"{var}_bacia"] = np.nan

diaria = cal.set_index("data").join([w_v, w_f, w_c]).reset_index()
ordem = (list(cal.columns) + [f"vazao_{p}" for p in POSTOS] + [f"flag_{p}" for p in POSTOS]
         + ["chuva_bacia", "tmed_bacia", "tmax_bacia", "tmin_bacia"]
         + [f"{var}_{p}" for var in curto.values() for p in POSTOS])
diaria = diaria[ordem]
print(diaria.shape)
diaria.tail(3)

(685, 42)


,data,ano,mes,dia_semana,dia_ano,fim_de_semana,estacao,feriado_nacional,feriado_estadual,ponto_facultativo,nome_feriado,feriado,dia_util,ponte,vazao_porto_amazonas,vazao_rio_negro,vazao_fluviopolis,vazao_uniao_da_vitoria,flag_porto_amazonas,flag_rio_negro,flag_fluviopolis,flag_uniao_da_vitoria,chuva_bacia,tmed_bacia,tmax_bacia,tmin_bacia,chuva_porto_amazonas,chuva_rio_negro,chuva_fluviopolis,chuva_uniao_da_vitoria,tmed_porto_amazonas,tmed_rio_negro,tmed_fluviopolis,tmed_uniao_da_vitoria,tmax_porto_amazonas,tmax_rio_negro,tmax_fluviopolis,tmax_uniao_da_vitoria,tmin_porto_amazonas,tmin_rio_negro,tmin_fluviopolis,tmin_uniao_da_vitoria
682,2026-11-14,2026,11,5,318,True,primavera,False,False,False,,False,False,False,117.199997,218.300003,1200.000000,1597.900024,ok,ok,ok,ok,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
683,2026-11-15,2026,11,6,319,True,primavera,True,False,False,Our Lady of Rocio; Republic Proclamation Day,True,False,False,111.199997,186.800003,1196.500000,1542.400024,ok,ok,ok,ok,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
684,2026-11-16,2026,11,0,320,False,primavera,False,False,False,,False,True,False,109.699997,179.600006,1187.699951,1532.699951,ok,ok,ok,ok,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 8. Agregação mensal (P2 – estiagens)
Os dados diários são agregados por mês:
- **Vazão:** média, mínima e máxima do mês, por posto.
- **Chuva:** soma do mês (total de chuva na bacia).
- **Temperatura:** média do mês.
- **Feriados:** número de feriados e de dias úteis no mês.

In [52]:
# vazão: média, mínima e máxima do mês por posto
v_mes = (vazao.assign(mes_ref=vazao["data"].dt.to_period("M").dt.to_timestamp())
              .groupby(["mes_ref", "posto"])["vazao"]
              .agg(vazao_media="mean", vazao_min="min", vazao_max="max")
              .unstack("posto")
              .reindex(columns=list(POSTOS), level="posto"))
v_mes.columns = [f"{met}_{p}" for met, p in v_mes.columns]

# clima e calendário: chuva somada, temperatura média, feriados contados
c_mes = (diaria.assign(mes_ref=diaria["data"].dt.to_period("M").dt.to_timestamp())
               .groupby("mes_ref")
               .agg(chuva_bacia_mm=("chuva_bacia", "sum"),
                    tmed_bacia=("tmed_bacia", "mean"),
                    tmax_bacia=("tmax_bacia", "mean"),
                    tmin_bacia=("tmin_bacia", "mean"),
                    n_feriados=("feriado", "sum"),
                    n_dias_uteis=("dia_util", "sum")))

mensal = c_mes.join(v_mes).reset_index()
mensal.insert(1, "ano", mensal["mes_ref"].dt.year)
mensal.insert(2, "mes", mensal["mes_ref"].dt.month)
print(mensal.shape)
mensal.tail(3)

(23, 21)


,mes_ref,ano,mes,chuva_bacia_mm,tmed_bacia,tmax_bacia,tmin_bacia,n_feriados,n_dias_uteis,vazao_media_porto_amazonas,vazao_media_rio_negro,vazao_media_fluviopolis,vazao_media_uniao_da_vitoria,vazao_min_porto_amazonas,vazao_min_rio_negro,vazao_min_fluviopolis,vazao_min_uniao_da_vitoria,vazao_max_porto_amazonas,vazao_max_rio_negro,vazao_max_fluviopolis,vazao_max_uniao_da_vitoria
20,2026-09-01,2026,9,208.175,16.010000,21.744167,11.785833,1,21,NaN,NaN,NaN,1844.011747,NaN,NaN,NaN,1257.599976,NaN,NaN,NaN,2221.762451
21,2026-10-01,2026,10,254.300,16.927419,22.150806,13.176613,1,21,294.066661,260.433334,1701.550018,1842.504497,248.100006,228.300003,1618.699951,1480.099976,339.799988,293.200012,1756.800049,2155.399902
22,2026-11-01,2026,11,66.625,18.845455,24.411364,14.604545,2,10,153.281248,238.837500,1384.587494,1943.424988,109.699997,150.899994,1187.699951,1532.699951,211.399994,367.600006,1742.599976,2194.300049


## 9. Validações

In [53]:
assert diaria["data"].is_unique and diaria["data"].diff().dropna().eq(pd.Timedelta(days=1)).all()
assert mensal["mes_ref"].is_unique
for p in POSTOS:
    assert (diaria[f"vazao_{p}"].dropna() > 0).all()
assert (diaria["chuva_bacia"].dropna() >= 0).all()
for p in POSTOS:
    pct = diaria[f"vazao_{p}"].isna().mean() * 100
    print(f"{NOMES_BONITOS[p]:>17}: {pct:5.1f}% dos dias sem vazão")
print(f"{'chuva_bacia':>17}: {diaria['chuva_bacia'].isna().mean()*100:5.1f}% dos dias sem chuva")
print("Validações estruturais ok")

   Porto Amazonas:  97.2% dos dias sem vazão
        Rio Negro:  97.2% dos dias sem vazão
      Fluviópolis:  97.1% dos dias sem vazão
 União da Vitória:   0.0% dos dias sem vazão
      chuva_bacia:   0.7% dos dias sem chuva
Validações estruturais ok


## 9. Exportação (Parquet + DuckDB) e download
Formato *tidy*: cada linha é uma observação (dia ou mês) e cada coluna uma variável. Também exportamos as tabelas longas
(posto × dia) e um banco **DuckDB** para as consultas SQL da Etapa 3. Tudo vai num `.zip`.

In [54]:
import duckdb
saidas = {"base_diaria": diaria, "base_mensal": mensal, "vazao_diaria_long": vazao,
          "clima_diario_long": clima, "calendario_feriados": cal}
db = SAIDA / "iguacu.duckdb"; db.unlink(missing_ok=True)
with duckdb.connect(str(db)) as con:
    for nome, tab in saidas.items():
        arq = SAIDA / f"{nome}.parquet"
        tab.to_parquet(arq, index=False)
        con.execute(f"CREATE TABLE {nome} AS SELECT * FROM read_parquet('{arq.as_posix()}')")
    print(con.sql("SELECT table_name, estimated_size AS linhas FROM duckdb_tables() ORDER BY 1"))

relatorio = {"arquivos_ons": log_arquivos.to_dict("records"), "colunas_detectadas": COLS,
             "mapeamento_postos": mapeamento, "formatos_data": formatos, "periodo": [INICIO, FIM],
             "tratamento_vazao": rel, "tratamento_clima": rel_clima, "q90_ref": q90.to_dict()}
(SAIDA / "relatorio_preparacao.json").write_text(json.dumps(relatorio, indent=2, ensure_ascii=False, default=str))
qualidade.to_csv(SAIDA / "qualidade_vazao_por_posto.csv")

pd.testing.assert_frame_equal(pd.read_parquet(SAIDA / "base_diaria.parquet"), diaria, check_dtype=False)

zip_path = BASE / "etapa2_bases.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for f in SAIDA.iterdir():
        z.write(f, f.name)
print("Gerado:", zip_path)
if EM_COLAB:
    colab_files.download(str(zip_path))

┌─────────────────────┬────────┐
│     table_name      │ linhas │
│       varchar       │ int64  │
├─────────────────────┼────────┤
│ base_diaria         │    685 │
│ base_mensal         │     23 │
│ calendario_feriados │    685 │
│ clima_diario_long   │   2720 │
│ vazao_diaria_long   │   2740 │
└─────────────────────┴────────┘

Gerado: /content/etapa2_bases.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>